# 05 - Skew & Term Structure

- **Skew**: difference between OTM put IV and OTM call IV (25-delta)
- **Term structure**: short-dated IV minus long-dated IV

Both are regime indicators.


## Setup

In [ ]:
import sys, sqlite3
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_p = Path.cwd().resolve()
while _p != _p.parent and not (_p / "database").exists():
    _p = _p.parent
sys.path.insert(0, str(_p))

OPT_DB = _p / "database" / "options.db"
MKT_DB = _p / "database" / "market.db"

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (13, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

def opt(q, **p):
    with sqlite3.connect(OPT_DB) as c:
        return pd.read_sql_query(q, c, params=p)

def mkt(q, **p):
    with sqlite3.connect(MKT_DB) as c:
        return pd.read_sql_query(q, c, params=p)

print("options.db:", OPT_DB)
print("market.db :", MKT_DB)

## 1. Load summary

In [ ]:
s = opt("SELECT * FROM option_summary WHERE underlying='ETH' ORDER BY ts")
s["date_dt"] = pd.to_datetime(s.ts, unit="ms")
s = s.set_index("date_dt")

## 2. Skew & term charts

In [ ]:
if not s.empty:
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4))
    if s.iv_25d_skew.notna().any():
        a1.plot(s.index, s.iv_25d_skew * 100, "o-", color="purple")
        a1.axhline(0, ls="--", color="black", lw=0.8)
    a1.set_title("25-delta skew")
    a1.set_ylabel("%")

    if s.term_structure.notna().any():
        a2.plot(s.index, s.term_structure * 100, "o-", color="darkorange")
        a2.axhline(0, ls="--", color="black", lw=0.8)
    a2.set_title("term structure (7d - 30d)")
    a2.set_ylabel("%")
    plt.tight_layout()
    plt.show()

## 3. Compute skew from raw chain

In [ ]:
iv = opt("""
    SELECT * FROM implied_volatility
    WHERE underlying='ETH'
      AND ts = (SELECT MAX(ts) FROM implied_volatility WHERE underlying='ETH')
""")
oc = opt("SELECT underlying_px FROM option_chain ORDER BY ts DESC LIMIT 1")
spot = float(oc.underlying_px.iloc[0]) if not oc.empty else None

if spot:
    target_expiry = iv.expiry.iloc[0]
    snap = iv[iv.expiry == target_expiry].copy()
    near = snap.iloc[(snap.strike - spot).abs().argsort()[:5]]
    print("nearest 5 strikes to spot:")
    print(near[["strike", "kind", "iv"]].sort_values(["kind", "strike"]))